# Gemma 4 full DRC language evaluation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashuza11/CongoLangBench/blob/main/notebooks/gemma4_all_languages_full_evaluation.ipynb)

This is the production run: 47 languages × 1,500 pairs × two directions = **141,000 predictions**. Results checkpoint directly to private Google Drive storage and safely resume after a Colab interruption.

## Before starting

1. Select a 40 GB GPU runtime when available.
2. Accept access to `google/gemma-4-12B-it` and add `HF_TOKEN` under Colab Secrets.
3. Build `private_data/congolang-benchmark-v1.zip` locally with `venv/bin/python scripts/package_colab_benchmarks.py`.
4. Keep the uploaded benchmark and Drive results private: they include restricted text.
5. Rerun the notebook after an interruption; completed record IDs are skipped.

In [ ]:
REPO_URL = "https://github.com/Ashuza11/CongoLangBench.git"
REPO_BRANCH = "main"
BATCH_SIZE = 8  # Reduce only if automatic OOM recovery reaches single rows.
MAX_NEW_TOKENS = 512
RETRY_MAX_NEW_TOKENS = 768

In [ ]:
# Gemma 4 requires the multimodal auto-loader shipped by Transformers 5.
%pip uninstall -q -y gradio diffusers
%pip install -q -U "transformers>=5,<6" accelerate bitsandbytes huggingface_hub sacrebleu "pandas==2.2.3"

In [ ]:
import platform, subprocess, torch
from pathlib import Path
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")
gpu = torch.cuda.get_device_properties(0)
print(f"Python {platform.python_version()} | PyTorch {torch.__version__}")
print(f"GPU: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB)")

In [ ]:
REPO_ROOT = Path("/content/CongoLangBench")
if REPO_ROOT.exists():
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
print(subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip())

In [ ]:
from google.colab import drive, files
from huggingface_hub import login, notebook_login
drive.mount("/content/drive")
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
login(token=hf_token, add_to_git_credential=False) if hf_token else notebook_login()

## Upload the private benchmark

Upload exactly `congolang-benchmark-v1.zip`. It is extracted only in this temporary runtime; the runner verifies every frozen checksum before loading the model.

In [ ]:
import shutil, zipfile
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError(f"Upload exactly one ZIP; received {list(uploaded)}")
DATA_ROOT = Path("/content/congolang-benchmark-private")
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True)
with zipfile.ZipFile(zip_names[0]) as archive:
    for member in archive.infolist():
        destination = (DATA_ROOT / member.filename).resolve()
        if not destination.is_relative_to(DATA_ROOT.resolve()):
            raise ValueError(f"Unsafe ZIP member: {member.filename}")
    archive.extractall(DATA_ROOT)
del uploaded

## Run or resume the complete evaluation

This cell may run for hours. Its output is appended to `MyDrive/CongoLangBench-private/gemma4-12b-it-full-v1/predictions.jsonl` after every batch. If Colab stops, reconnect, rerun the earlier cells, and execute this cell again.

In [ ]:
OUTPUT_ROOT = Path("/content/drive/MyDrive/CongoLangBench-private/gemma4-12b-it-full-v1")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
command = [
    "python", "-u", str(REPO_ROOT / "scripts/run_gemma_full.py"),
    "--repo-root", str(REPO_ROOT),
    "--data-root", str(DATA_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--batch-size", str(BATCH_SIZE),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--retry-max-new-tokens", str(RETRY_MAX_NEW_TOKENS),
]
subprocess.run(command, check=True)

In [ ]:
import json
metadata_path = OUTPUT_ROOT / "run_metadata.json"
if not metadata_path.is_file():
    raise RuntimeError("The full run is not complete yet. Rerun later to resume.")
metadata = json.loads(metadata_path.read_text())
assert metadata["requests"] == 141000
score_command = [
    "python", str(REPO_ROOT / "scripts/score_full_run.py"),
    "--repo-root", str(REPO_ROOT),
    "--data-root", str(DATA_ROOT),
    "--run-root", str(OUTPUT_ROOT),
    "--output-root", str(OUTPUT_ROOT / "scored"),
]
subprocess.run(score_command, check=True)
print(json.dumps(metadata, indent=2))
print("Gemma full evaluation and aggregate scoring are complete in private Drive storage.")